# 02 — Features: scaled numerics + sector-stratified design matrix

**Project:** Dynamic Retention Benchmarking (H03)

The Ridge fits run on four scaled numeric columns: `headcount`, `comp_percentile`, `training_hours`, `manager_quality`. We exercise `build_pipeline()` here and produce diagnostics that justify the modelling choices in `03_model.ipynb`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from retention_bench.data import generate, write_processed, PROCESSED
from retention_bench.features import NUMERIC_FEATURES, add_engineered, build_pipeline

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
PARQUET = PROCESSED / 'retention_panel.parquet'
df = pd.read_parquet(PARQUET) if PARQUET.exists() else generate()
df = add_engineered(df)
df.head()

## 1. Pearson correlation between predictors and target

Per-sector Ridge weights interpret these correlations conditioned on sector — but the marginal table is a useful first-cut sanity check.

In [ ]:
C = df[NUMERIC_FEATURES + ['retention_rate']].corr()
fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(C.round(2), annot=True, cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Marginal Pearson correlations')
plt.tight_layout(); plt.show()

## 2. Apply the production preprocessor

`build_pipeline()` is a `ColumnTransformer` that StandardScales the four numeric columns. We confirm the resulting matrix has zero mean and unit variance.

In [ ]:
pipe = build_pipeline()
X = pipe.fit_transform(df[NUMERIC_FEATURES])
print('shape:', X.shape)
pd.DataFrame(X, columns=NUMERIC_FEATURES).describe().round(2)

## 3. Within-sector scaling check

Because we fit one Ridge *per sector*, each per-sector preprocessor will see its own scaler statistics. We sanity-check that within-sector variance is non-trivial — a sector with constant `manager_quality` would collapse the ridge slope to NaN.

In [ ]:
stats = df.groupby('sector')[NUMERIC_FEATURES].agg(['mean', 'std']).round(2)
stats

## 4. Pairwise scatter — within-sector covariance

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.scatterplot(data=df, x='comp_percentile', y='training_hours', hue='sector', alpha=0.5, ax=ax)
ax.set_title('comp_percentile vs training_hours, conditioned on sector')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

## 5. Action-as-context view

The bandit treats the action as the arm and the row's predictors as context. The featurisation is identical for the Ridge and the bandit — neither reads `action_taken` as a *feature*; the bandit only needs it as the arm label.

In [ ]:
g = df.groupby('action_taken')[NUMERIC_FEATURES].mean().round(2)
g

## 6. Train-time bookkeeping

The Ridge fits per sector; the bandit fits across sectors. We list, per sector, the sample size — Ridge with `n < 100` would warrant a fallback to the global model.

In [ ]:
n_per_sector = df.groupby('sector').size().sort_values()
fig, ax = plt.subplots(figsize=(8, 3.4))
n_per_sector.plot(kind='bar', ax=ax, color='#3a7ca5')
ax.axhline(100, color='red', ls='--', label='Ridge fallback floor')
ax.set_title('Training rows per sector')
ax.legend()
plt.tight_layout(); plt.show()

## 7. Hand-off

The featurisation choices are conservative because the load-bearing structure here is the *per-sector* model; richer non-linearities would over-fit at this n. `03_model.ipynb` fits one `Ridge(alpha=1.0)` per sector and a global `EpsilonGreedyBandit` over the discrete action set.